# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
# Two signal checks BEFORE the rule. Both signals sit behind real FlyRank flags:
#   Signal 1 = staleness       -> behind the refresh flags
#   Signal 2 = CTR-vs-position -> behind the CTR-fix logic
import os, json
from pathlib import Path
import pandas as pd, numpy as np

if Path.cwd().name == "notebooks":
    os.chdir("../..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["declining"] = df["trend_direction"].str.lower().eq("down").astype(int)
BASE_RATE = df["declining"].mean()
print(f"n = {len(df):,} pages | base rate (declining) = {BASE_RATE:.3f}\n")

# ---- SIGNAL 1: staleness. Claim behind the refresh flag: staler pages decline more. ----
print("=" * 68)
print("SIGNAL 1 - staleness (behind the refresh flags)")
print("=" * 68)
s1 = (df.groupby("freshness_tier", observed=True)["declining"]
        .agg(n="size", decline_rate="mean")
        .reindex(["0-30", "31-90", "91-180", "181+"]))
s1["vs_base"] = s1["decline_rate"] - BASE_RATE
print(s1.round(3).to_string())
print("\nVERDICT: MIXED")
print("  Right direction where the data lives: 0-30d sits below base and 91-180d above,")
print("  and those two buckets hold 29,651 of 30,000 pages.")
print("  But the STALEST bucket, 181+, has the LOWEST decline rate of all.")
print("  'Staler = worse' does not survive its own tail, and the 31-90 / 181+ buckets")
print("  carry n=175 and n=174 -- too thin to lean on.")

# ---- SIGNAL 2: CTR vs position. Claim behind CTR-fix: low CTR for its rank -> decline. ----
print("\n" + "=" * 68)
print("SIGNAL 2 - CTR vs position (behind the CTR-fix logic)")
print("=" * 68)
# avg_position == 0 means NO DATA, not rank zero. Require volume so CTR is stable.
v = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 100)].copy()
v["pos_band"] = pd.cut(v["avg_position"], [0, 10, 20, 1000], labels=["1-10", "11-20", "21+"])
v["ctr_vs_band"] = np.where(
    v["ctr"] < v.groupby("pos_band", observed=True)["ctr"].transform("median"),
    "below median for its band", "at/above median")
s2 = (v.groupby(["pos_band", "ctr_vs_band"], observed=True)["declining"]
        .agg(n="size", decline_rate="mean"))
print(f"eligible rows (avg_position > 0, impressions >= 100): {len(v):,}\n")
print(s2.round(3).to_string())
print("\nVERDICT: MIXED")
print("  Positions 1-10:  below-band CTR declines MORE (+0.15)")
print("  Positions 11-20: same direction (+0.11)")
print("  Positions 21+:   REVERSES (-0.06)")
print("  The CTR-fix logic is real on page 1-2 and backwards past position 20.")
print("  That negative is the useful part: it tells me where NOT to apply the rule.")


n = 30,000 pages | base rate (declining) = 0.542

SIGNAL 1 - staleness (behind the refresh flags)
                    n  decline_rate  vs_base
freshness_tier                              
0-30            20480         0.511   -0.031
31-90             175         0.589    0.047
91-180           9171         0.611    0.069
181+              174         0.471   -0.071

VERDICT: MIXED
  Right direction where the data lives: 0-30d sits below base and 91-180d above,
  and those two buckets hold 29,651 of 30,000 pages.
  But the STALEST bucket, 181+, has the LOWEST decline rate of all.
  'Staler = worse' does not survive its own tail, and the 31-90 / 181+ buckets
  carry n=175 and n=174 -- too thin to lean on.

SIGNAL 2 - CTR vs position (behind the CTR-fix logic)
eligible rows (avg_position > 0, impressions >= 100): 22,006

                                       n  decline_rate
pos_band ctr_vs_band                                  
1-10     at/above median            4635         0.539
     

### The rule, in plain words

> **A page is worth reviewing first if it is still visible in search, and it is underperforming on click-through *for the position band it actually ranks in*, but only where that logic held up (positions 1–20). If it is visible and stale instead, it is a refresh candidate. Everything else is monitored, and pages nobody sees are skipped.**

Both signal checks came back **MIXED**, and both negatives changed the rule rather than decorating it:

- **Signal 2 reversed past position 20**, so the CTR flag is *restricted to positions 1–20*. Applying it deeper would have ranked pages by a relationship that runs the other way.
- **Signal 1's tail reversed too**, so staleness is a **secondary** reason (weight 2, not 3) and uses the shipped `91-180` tier, where the effect was actually measured, not the sparse `181+` bucket where it inverts.

### The reason codes

One page gets exactly **one** reason code, first match wins. Priority is the order below.

| Priority | Reason code | Action | Weight | Condition |
|---|---|---|---|---|
| n/a | `not_visible` | `SKIP` | 0 | `impressions_90d < 500`, so nobody sees it and no edit pays off |
| 1 | `ctr_below_band_top20` | `FIX_CTR` | 3 | Ranks 1–20 **and** CTR below its band median, the strongest measured segment |
| 2 | `stale_and_visible` | `REFRESH` | 2 | Freshness tier `91-180`, the window where staleness tracked decline |
| 3 | `deep_position` | `MONITOR` | 1 | Ranks 21+, deliberately *not* CTR-flagged, because Signal 2 reverses here |
| 4 | `visible_no_flag` | `MONITOR` | 1 | Visible, no flag fired |

### The score

```
score = weight × log10(impressions_90d) × (1 + ctr_shortfall)
```

Three readable terms, no fitted weights: **which flag fired** (weight), **how much traffic is at stake** (log10, so a 500k-impression page does not simply buy the top spot), and **how far below its band the CTR sits** (shortfall). Section 4 reports what this ordering costs.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
VISIBLE_MIN = 500          # "still visible in search"
q = df.copy()

# --- the conditions, each one readable on its own ---------------------------
q["visible"]      = q["impressions_90d"] >= VISIBLE_MIN
q["has_position"] = q["avg_position"] > 0          # 0 = NO DATA, not rank zero
q["top20"]        = q["has_position"] & (q["avg_position"] <= 20)
q["pos_band"]     = pd.cut(q["avg_position"].where(q["has_position"]),
                           [0, 10, 20, 1000], labels=["1-10", "11-20", "21+"])
band_median_ctr   = q.groupby("pos_band", observed=True)["ctr"].transform("median")
q["ctr_shortfall"] = (band_median_ctr - q["ctr"]).clip(lower=0).fillna(0)
q["ctr_below_band"] = q["ctr"] < band_median_ctr
q["stale"]        = q["freshness_tier"].eq("91-180")

# --- ONE reason code per page, first match wins -----------------------------
def classify(r):
    if not r.visible:                    return ("not_visible",          "SKIP",    0)
    if r.top20 and r.ctr_below_band:     return ("ctr_below_band_top20", "FIX_CTR", 3)
    if r.stale:                          return ("stale_and_visible",    "REFRESH", 2)
    if r.has_position and not r.top20:   return ("deep_position",        "MONITOR", 1)
    return                                      ("visible_no_flag",      "MONITOR", 1)

q[["reason_code", "action", "weight"]] = q.apply(classify, axis=1, result_type="expand")

# --- the score --------------------------------------------------------------
q["score"] = (q["weight"]
              * np.log10(q["impressions_90d"].clip(lower=1))
              * (1 + q["ctr_shortfall"]))
q = q.sort_values("score", ascending=False).reset_index(drop=True)
q["rank"] = q.index + 1

print("reason code mix (n and the decline rate each segment actually had):")
print(q.groupby(["reason_code", "action"], observed=True)
       .agg(n=("score", "size"), decline_rate=("declining", "mean"))
       .sort_values("n", ascending=False).round(3).to_string())

# --- evaluate at K, always next to the base rate ----------------------------
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

y = q["declining"].values
p_at = {k: precision_at_k(q["score"], y, k) for k in (10, 20, 50, 100)}
print(f"\nbase rate = {BASE_RATE:.3f}   <- the bar. Below this, shuffling would do better.")
for k, val in p_at.items():
    flag = "beats base" if val > BASE_RATE else "BELOW base"
    print(f"  precision@{k:<4} = {val:.3f}   ({flag})")

# --- write the queue (CSV is gitignored by design; it regenerates every run) --
OUT = Path("work/outputs")
OUT.mkdir(parents=True, exist_ok=True)
cols = ["rank", "content_id", "client_id", "score", "action", "reason_code",
        "impressions_90d", "avg_position", "ctr", "ctr_shortfall",
        "days_since_last_update", "freshness_tier"]
q[cols].to_csv(OUT / "baseline_action_score.csv", index=False)
print(f"\nwrote {OUT / 'baseline_action_score.csv'}  ({len(q):,} rows)")

# --- metrics JSON: the run's receipts, and this one IS committed ------------
metrics = {
    "rule": "weight * log10(impressions_90d) * (1 + ctr_shortfall)",
    "visible_min_impressions": VISIBLE_MIN,
    "n_pages": int(len(q)),
    "base_rate": round(float(BASE_RATE), 4),
    "precision_at_k": {str(k): round(v, 4) for k, v in p_at.items()},
    "signal_verdicts": {"staleness": "MIXED", "ctr_vs_position": "MIXED"},
    "reason_code_counts": q["reason_code"].value_counts().to_dict(),
    "action_counts": q["action"].value_counts().to_dict(),
}
(OUT / "baseline_metrics.json").write_text(json.dumps(metrics, indent=2))
print(f"wrote {OUT / 'baseline_metrics.json'}")


reason code mix (n and the decline rate each segment actually had):
                                  n  decline_rate
reason_code          action                      
not_visible          SKIP     13274         0.475
visible_no_flag      MONITOR   5487         0.551
stale_and_visible    REFRESH   5081         0.579
ctr_below_band_top20 FIX_CTR   3673         0.701
deep_position        MONITOR   2485         0.571

base rate = 0.542   <- the bar. Below this, shuffling would do better.
  precision@10   = 0.600   (beats base)
  precision@20   = 0.500   (BELOW base)
  precision@50   = 0.500   (BELOW base)
  precision@100  = 0.460   (BELOW base)



wrote work\outputs\baseline_action_score.csv  (30,000 rows)
wrote work\outputs\baseline_metrics.json


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# TOP-10 REVIEW. One line each: the action, why it is there, what would make it wrong.
# The "what would make it wrong" text is derived from each row's own values, so it
# stays true if the data or the rule changes.
top10 = q.head(10)

def why_here(r):
    return {
        "ctr_below_band_top20": (f"ranks {r.avg_position:.1f} (band {r.pos_band}) with CTR "
                                 f"{r.ctr:.2f}%, {r.ctr_shortfall:.2f}pp below its band median"),
        "stale_and_visible":    f"not updated in {r.days_since_last_update:.0f} days, still visible",
        "deep_position":        f"ranks {r.avg_position:.1f}, too deep for the CTR flag",
        "visible_no_flag":      "visible but no flag fired",
        "not_visible":          "below the visibility floor",
    }[r.reason_code]

def what_would_make_it_wrong(r):
    reasons = []
    if r.ctr == 0 and r.clicks_90d == 0:
        reasons.append("zero clicks in 90d may be a tracking gap, not a real CTR problem")
    if r.avg_position <= 3:
        reasons.append(f"already ranks {r.avg_position:.1f} -- little headroom, low CTR may be "
                       "structural (the answer shows in the SERP)")
    if r.impressions_90d > 200_000:
        reasons.append("very large page: low CTR is often structural for broad informational "
                       "queries, not a fixable defect")
    if r.freshness_tier == "0-30":
        reasons.append(f"updated {r.days_since_last_update:.0f} days ago -- someone may have "
                       "just worked on it")
    if r.ctr_shortfall < 0.05:
        reasons.append("shortfall is tiny; it barely cleared the median test")
    return "; ".join(reasons) if reasons else \
           "a rewrite that changes intent could lose the impressions it already has"

print(f"{'#':<3}{'ACTION':<9}{'IMPR':>9}{'POS':>6}{'CTR':>6}  REVIEW")
print("=" * 118)
for _, r in top10.iterrows():
    print(f"{r['rank']:<3}{r.action:<9}{r.impressions_90d:>9,}{r.avg_position:>6.1f}{r.ctr:>6.2f}")
    print(f"      why: {why_here(r)}")
    print(f"      wrong if: {what_would_make_it_wrong(r)}")
    print("-" * 118)

hits = int(top10["declining"].sum())
print(f"\nOf these top 10, {hits} were actually declining "
      f"(precision@10 = {hits/10:.2f} vs base rate {BASE_RATE:.3f}).")


#  ACTION        IMPR   POS   CTR  REVIEW
1  FIX_CTR    208,678   9.7  0.00
      why: ranks 9.7 (band 1-10) with CTR 0.00%, 0.15pp below its band median
      wrong if: zero clicks in 90d may be a tracking gap, not a real CTR problem; very large page: low CTR is often structural for broad informational queries, not a fixable defect
----------------------------------------------------------------------------------------------------------------------
2  FIX_CTR    272,144   2.3  0.03
      why: ranks 2.3 (band 1-10) with CTR 0.03%, 0.12pp below its band median
      wrong if: already ranks 2.3 -- little headroom, low CTR may be structural (the answer shows in the SERP); very large page: low CTR is often structural for broad informational queries, not a fixable defect; updated 20 days ago -- someone may have just worked on it
----------------------------------------------------------------------------------------------------------------------
3  FIX_CTR    295,097   7.3  0.05
      why: 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# --- LEAKAGE CHECK: assert it, don't claim it -------------------------------
LABEL_DERIVED = {"trend_direction", "trend_pct", "declining", "is_declining_label"}
RULE_INPUTS = {"impressions_90d", "avg_position", "ctr", "freshness_tier",
               "days_since_last_update", "clicks_90d"}

leaked = RULE_INPUTS & LABEL_DERIVED
assert not leaked, f"LEAK: {leaked}"
print("LEAKAGE CHECK")
print(f"  rule inputs        : {sorted(RULE_INPUTS)}")
print(f"  label-derived cols : {sorted(LABEL_DERIVED)}")
print(f"  overlap            : {sorted(leaked) or 'NONE'}  -> no label-derived input\n")

# No future windows: every input is a trailing-90d or as-of-today measurement.
# No product decision flags: the starter data ships none, so none can slip in.
print("  future windows     : none - all inputs are trailing-90d or as-of-today")
print("  product flags      : none - the starter export contains no decision flags")
print("  IDs as features    : none - content_id/client_id are output columns only\n")

# --- WEAK PICKS: the rule's own failures, measured --------------------------
print("=" * 70)
print("WEAK PICKS")
print("=" * 70)
misses = q.head(50)[q.head(50)["declining"] == 0]
print(f"Of the top 50, {len(misses)} were NOT declining. Their shared shape:\n")
print(misses[["impressions_90d", "avg_position", "ctr", "days_since_last_update"]]
      .describe().loc[["count", "min", "50%", "max"]].round(2).to_string())

# Why the ordering hurts: does traffic size itself track decline?
print("\nDecline rate by impression quintile (the mechanism):")
print(q.assign(imp_q=pd.qcut(q["impressions_90d"], 5,
                             labels=["Q1 low", "Q2", "Q3", "Q4", "Q5 high"]))
       .groupby("imp_q", observed=True)["declining"]
       .agg(n="size", decline_rate="mean").round(3).to_string())
print(f"\nbase rate = {BASE_RATE:.3f}")
print("Q5 (the biggest pages) declines LESS than Q2-Q4. Any 'biggest pages first'")
print("ordering therefore fights the label -- which is exactly what my score does.")


LEAKAGE CHECK
  rule inputs        : ['avg_position', 'clicks_90d', 'ctr', 'days_since_last_update', 'freshness_tier', 'impressions_90d']
  label-derived cols : ['declining', 'is_declining_label', 'trend_direction', 'trend_pct']
  overlap            : NONE  -> no label-derived input

  future windows     : none - all inputs are trailing-90d or as-of-today
  product flags      : none - the starter export contains no decision flags
  IDs as features    : none - content_id/client_id are output columns only

WEAK PICKS
Of the top 50, 25 were NOT declining. Their shared shape:

       impressions_90d  avg_position    ctr  days_since_last_update
count             25.0          25.0  25.00                    25.0
min            44860.0           2.2   0.01                     7.0
50%           111222.0           6.7   0.06                    22.0
max           295097.0           9.0   0.12                   106.0

Decline rate by impression quintile (the mechanism):
            n  decline_rat

### What the weak picks are telling me

**The honest headline: my rule beats the base rate at K=10 and not at K=50.** The segment logic is sound, since `ctr_below_band_top20` really does decline at ~0.70 against a 0.542 base, but the *ordering within it* is working against me.

The mechanism is in the quintile table above: the largest pages decline **less** than mid-sized ones. So multiplying by traffic, even as `log10`, pulls exactly the wrong pages to the top. The flag picks a good pool; the sort then picks the worst members of it.

**I tested the alternatives rather than guessing.** Ranking by CTR shortfall alone reaches precision@50 ≈ 0.94, and I did not ship it, because the top 50 are all *tied* at the maximum shortfall (zero-click pages ranking 1–10 with 500–7,000 impressions). The order inside that tie is arbitrary, so a different sort seed gives a different 50 and a different score. It is a number that would not survive being re-run, and it points editors at pages with almost no traffic at stake.

That is the real tension in this lane, and it is worth stating plainly: **precision@50 and business value pull in opposite directions here.** The queue that scores best is a list of tiny pages nobody would thank you for fixing.

### The two weakest picks in the top 10

- **Rank 2** (272k impressions, position 2.3, CTR 0.03%) is not declining. It already ranks near the top, so there is little headroom, and at that size a low CTR is usually structural: broad informational queries where the answer is visible in the SERP. The flag cannot tell "underperforming" from "this is just what a big page looks like".
- **Rank 7** (517k impressions, CTR 0.14%) is the largest page in the queue, and its shortfall barely cleared the median test. It is here because of its size, not because of evidence.

### What I would change in ML-08

1. **Drop the traffic term from the ranking** and use it as a display column instead, so the editor sees what is at stake without it deciding the order.
2. **Add a minimum shortfall threshold** so pages that barely miss their band median do not enter on size alone.
3. **Let the model learn the interaction** between page size and CTR shortfall. This is precisely the "many signals, tangled" case I argued for in ML-03, and it is where the model should earn its place.

The baseline is now **frozen** at these numbers. Moving it later to flatter the model would convince nobody, including me.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere: pseudonymous IDs only
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Written from this notebook:** `work/outputs/baseline_action_score.csv` (the ranked queue, gitignored by design, regenerates on every run) and `work/outputs/baseline_metrics.json` (the run's receipts, and this one **is** committed).
